In [ ]:
import pandas as pd

df = pd.read_csv('rahul_transactions.csv')

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (1328, 8)


In [ ]:
df['date'] = pd.to_datetime(
    df['Date'],
    errors='coerce',
    dayfirst=True,
    format='mixed'
)

df['amount'] = (
    df['Amount']
    .astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace('Rs.', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)

df['amount'] = pd.to_numeric(df['amount'], errors='coerce')

df['type_clean'] = (
    df['Type'].astype(str).str.lower().str.strip()
)

df['type_clean'] = df['type_clean'].replace({
    'dr': 'debit',
    'debit': 'debit',
    'cr': 'credit',
    'credit': 'credit'
})

df = df.drop_duplicates().copy()

print("Cleaning completed!")
print("Shape:", df.shape)

Cleaning completed!
Shape: (1310, 11)


In [ ]:
def extract_vendor(description):
    ...

In [ ]:
df['vendor_clean'] = df['Description'].apply(extract_vendor)

print("Vendor extraction completed!")

Vendor extraction completed!


In [ ]:
print("Uncategorised transactions:",
      (df['vendor_clean'] == 'Uncategorised').sum())

Uncategorised transactions: 0


In [ ]:
category_keywords = {

    # Food & Dining
    'Food & Dining': [
        'SWIGGY',
        'BUNDL TECH P L',
        'ZOMATO',
        'INSTAMART',
        'RESTAURANT',
        'EMPIRE RESTAURANT',
        'MEGHANA FOODS',
        'DINEOUT',
        'STARBUCKS',
        'THIRDWAVE',
        'THIRD WAVE',
        'TWC INDIA',
        'COFFEE DAY',
        'TRUFFLES'
    ],

    # Shopping
    'Shopping': [
        'AMAZON',
        'AMZN',
        'FLIPKART',
        'FKART',
        'MYNTRA',
        'NYKAA',
        'FSN E-COMMERCE',
        'BLINKIT',
        'ZEPTO',
        'KIRANAKART',
        'BIGBASKET',
        'GROFERS',
        'DMART',
        'AVENUE SUPERMARTS',
        'INNOVATIVE RETAIL'
    ],

    # Transport
    'Transport': [
        'UBER',
        'ANI TECHNOLOGIES',
        'OLA CABS',
        'OLA-PRIME',
        'OLA ELECTRIC',
        'UPI-OLACABS',
        'RAPIDO',
        'BMTC',
        'TUMMOC',
        'ROPPEN TRANSPORTATION'
    ],

    # Fuel
    'Fuel': [
        'HP PETROL',
        'BPCL',
        'INDIAN OIL',
        'IOC'
    ],

    # Utilities
    'Utilities': [
        'BESCOM',
        'BANGALORE ELEC SUPPLY',
        'BWSSB',
        'RELIANCE JIO',
        'JIORECHARGE',
        'JIOFIBER',
        'AIRTEL',
        'BHARTI AIRTEL',
        'VODAFONE IDEA',
        'VI POSTPAID',
        'UPI-VI-RECHARGE'
    ],

    # Entertainment
    'Entertainment': [
        'NETFLIX',
        'SPOTIFY',
        'HOTSTAR',
        'DISNEY HOTSTAR',
        'BOOKMYSHOW',
        'BMS MOVIE TICKETS',
        'STAR INDIA PVT LTD',
        'BIGTREE ENTERTAINMENT'
    ],

    # Investment
    'Investment': [
        'GROWW',
        'ZERODHA'
    ],

    # Income
    'Income': [
        'SALARY',
        'TECHCRUSH LABS-SALARY'
    ],

    # Rent
    'Rent': [
        'RENT-LANDLORD'
    ],

    # Cash Withdrawal
    'Cash Withdrawal': [
        'ATM-WDL'
    ],

    # P2P Transfer
    'P2P Transfer': [
        'AMAN',
        'ANKIT',
        'PRIYA',
        'NEHA',
        'VIKAS',
        'SNEHA',
        'KARAN'
    ]
}

In [ ]:
def assign_category(description):

    text = str(description).upper().strip()

    for category, keywords in category_keywords.items():
        for keyword in keywords:
            if keyword in text:
                return category

    return 'Other'

In [ ]:
df['category'] = df['Description'].apply(assign_category)

print("Category tagging completed!")

Category tagging completed!


In [ ]:
print("Number of categories:", df['category'].nunique())

print("\nCategory counts:")
print(df['category'].value_counts())

Number of categories: 12

Category counts:
category
Food & Dining      527
Shopping           339
Transport          250
Entertainment       44
Utilities           43
Fuel                28
Investment          23
P2P Transfer        18
Cash Withdrawal     17
Other                9
Income               6
Rent                 6
Name: count, dtype: int64


In [ ]:
total_spending = df.loc[
    df['type_clean'] == 'debit',
    'amount'
].sum()

print("Total Spending: ₹", round(total_spending, 2))

Total Spending: ₹ 1678901.0


In [ ]:
average_transaction = df['amount'].mean()

print("Average Transaction: ₹", round(average_transaction, 2))

Average Transaction: ₹ 1670.74


In [ ]:
highest_transaction = df['amount'].max()

print("Highest Transaction: ₹", round(highest_transaction, 2))

Highest Transaction: ₹ 85492.0


In [ ]:
total_income = df.loc[
    df['type_clean'] == 'credit',
    'amount'
].sum()

print("Total Income: ₹", round(total_income, 2))

Total Income: ₹ 509774.0


In [ ]:
category_spending = (
    df[df['type_clean'] == 'debit']
    .groupby('category')['amount']
    .sum()
    .sort_values(ascending=False)
)

print("Category-wise Spending:")
print(category_spending)

Category-wise Spending:
category
Shopping           728338.0
Food & Dining      306642.0
Investment         248160.0
Rent               108000.0
Fuel                89303.0
Transport           57474.0
Cash Withdrawal     45500.0
Utilities           41914.0
Entertainment       26764.0
P2P Transfer        24599.0
Other                2207.0
Name: amount, dtype: float64


In [ ]:
print("=" * 50)
print("              SPENDING OVERVIEW")
print("=" * 50)

print(f"Total Spending       : ₹{total_spending:,.2f}")
print(f"Total Income         : ₹{total_income:,.2f}")
print(f"Average Transaction  : ₹{average_transaction:,.2f}")
print(f"Highest Transaction  : ₹{highest_transaction:,.2f}")

print("\nCategory-wise Spending:")
print(category_spending)

print("=" * 50)

              SPENDING OVERVIEW
Total Spending       : ₹1,678,901.00
Total Income         : ₹509,774.00
Average Transaction  : ₹1,670.74
Highest Transaction  : ₹85,492.00

Category-wise Spending:
category
Shopping           728338.0
Food & Dining      306642.0
Investment         248160.0
Rent               108000.0
Fuel                89303.0
Transport           57474.0
Cash Withdrawal     45500.0
Utilities           41914.0
Entertainment       26764.0
P2P Transfer        24599.0
Other                2207.0
Name: amount, dtype: float64


In [ ]:
# Make sure date column is in datetime format
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Extract month
df['month'] = df['date'].dt.to_period('M').astype(str)

# Calculate monthly debit spending
monthly_spending = (
    df[df['type_clean'] == 'debit']
    .groupby('month')['amount']
    .sum()
    .sort_index()
)

print("Monthly Spending:")
print(monthly_spending)

Monthly Spending:
month
2024-01    290767.0
2024-02    234071.0
2024-03    330458.0
2024-04    251382.0
2024-05    277306.0
2024-06    294917.0
Name: amount, dtype: float64


In [ ]:
monthly_transactions = (
    df[df['type_clean'] == 'debit']
    .groupby('month')
    .size()
    .sort_index()
)

print("Monthly Transaction Count:")
print(monthly_transactions)

Monthly Transaction Count:
month
2024-01    218
2024-02    212
2024-03    216
2024-04    221
2024-05    223
2024-06    214
dtype: int64


In [ ]:
print("=" * 50)
print("          MONTHLY SPENDING TREND")
print("=" * 50)

for month, amount in monthly_spending.items():
    print(f"{month} : ₹{amount:,.2f}")

print("=" * 50)

          MONTHLY SPENDING TREND
2024-01 : ₹290,767.00
2024-02 : ₹234,071.00
2024-03 : ₹330,458.00
2024-04 : ₹251,382.00
2024-05 : ₹277,306.00
2024-06 : ₹294,917.00


In [ ]:
def get_time_period(time_value):

    try:
        hour = int(str(time_value).split(':')[0])
    except:
        return 'Unknown'

    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    elif 21 <= hour or hour < 5:
        return 'Night'
    else:
        return 'Unknown'

In [ ]:
df['time_period'] = df['Time'].apply(get_time_period)

print("Time-of-day classification completed!")

Time-of-day classification completed!


In [ ]:
time_counts = df['time_period'].value_counts()

print("Transactions by Time of Day:")
print(time_counts)

Transactions by Time of Day:
time_period
Morning      354
Evening      340
Afternoon    314
Night        302
Name: count, dtype: int64


In [ ]:
time_spending = (
    df[df['type_clean'] == 'debit']
    .groupby('time_period')['amount']
    .sum()
    .sort_values(ascending=False)
)

print("Spending by Time of Day:")
print(time_spending)

Spending by Time of Day:
time_period
Morning      501130.0
Afternoon    452169.0
Evening      363884.0
Night        361718.0
Name: amount, dtype: float64


In [ ]:
print("=" * 50)
print("          TIME-OF-DAY ANALYSIS")
print("=" * 50)

print("\nTransaction Count:")
print(time_counts)

print("\nSpending:")
for period, amount in time_spending.items():
    print(f"{period:<12} : ₹{amount:,.2f}")

print("=" * 50)

          TIME-OF-DAY ANALYSIS

Transaction Count:
time_period
Morning      354
Evening      340
Afternoon    314
Night        302
Name: count, dtype: int64

Spending:
Morning      : ₹501,130.00
Afternoon    : ₹452,169.00
Evening      : ₹363,884.00
Night        : ₹361,718.00


In [ ]:
# Anomaly Detection using within-category Z-score

debit_df = df[df['type_clean'] == 'debit'].copy()

# Calculate category-wise mean and standard deviation
grouped = debit_df.groupby('category')['amount']

debit_df['category_mean'] = grouped.transform('mean')
debit_df['category_std'] = grouped.transform('std')

# Calculate Z-score
debit_df['z_score'] = (
    (debit_df['amount'] - debit_df['category_mean']) /
    debit_df['category_std'].replace(0, pd.NA)
)

# Identify anomalies
anomalies = debit_df[debit_df['z_score'] > 2].copy()

print("Anomaly Detection completed!")
print("Number of anomalies:", len(anomalies))

Anomaly Detection completed!
Number of anomalies: 27


In [ ]:
# Display detected anomalies

print("=" * 70)
print("                    ANOMALY TRANSACTIONS")
print("=" * 70)

print("\nTotal anomalies:", len(anomalies))

print("\nAnomaly Details:")
print(
    anomalies[
        ['date', 'Description', 'category', 'amount', 'category_mean', 'z_score']
    ].sort_values('z_score', ascending=False).to_string(index=False)
)

print("=" * 70)

                    ANOMALY TRANSACTIONS

Total anomalies: 27

Anomaly Details:
      date              Description      category  amount  category_mean    z_score
2024-02-26 POS BANGALORE RESTAURANT Food & Dining  8383.0     581.863378  10.017975
2024-06-22              POS DINEOUT Food & Dining  7935.0     581.863378   9.442668
2024-03-31        POS MEGHANA FOODS Food & Dining  7931.0     581.863378   9.437531
2024-03-04             POS TRUFFLES Food & Dining  7441.0     581.863378   8.808288
2024-01-07 POS BANGALORE RESTAURANT Food & Dining  7314.0     581.863378   8.645199
2024-06-26     AMAZONIN MARKETPLACE      Shopping 22008.0    2148.489676   5.638151
2024-02-07   UPI-AMAZONPAY@HDFCBANK      Shopping 21986.0    2148.489676   5.631905
2024-03-05   UPI-AMAZONPAY@HDFCBANK      Shopping 19917.0    2148.489676   5.044512
2024-03-02                AMAZON IN      Shopping 18273.0    2148.489676   4.577777
2024-05-27    UPI-FLIPKART@HDFCBANK      Shopping 17831.0    2148.489676   4.452

In [ ]:
# Spending Archetype Detection

# Calculate spending by category
category_spending = (
    df[df['type_clean'] == 'debit']
    .groupby('category')['amount']
    .sum()
    .sort_values(ascending=False)
)

# Total debit spending
total_spending = category_spending.sum()

# Calculate percentage of spending by category
category_percentage = (category_spending / total_spending) * 100

print("=" * 60)
print("             SPENDING ARCHETYPE DETECTION")
print("=" * 60)

print("\nCategory Spending Share:")

for category, percentage in category_percentage.items():
    print(f"{category:<20} : {percentage:.2f}%")

print("=" * 60)

             SPENDING ARCHETYPE DETECTION

Category Spending Share:
Shopping             : 43.38%
Food & Dining        : 18.26%
Investment           : 14.78%
Rent                 : 6.43%
Fuel                 : 5.32%
Transport            : 3.42%
Cash Withdrawal      : 2.71%
Utilities            : 2.50%
Entertainment        : 1.59%
P2P Transfer         : 1.47%
Other                : 0.13%


In [ ]:
# Determine Spending Archetype

top_category = category_percentage.idxmax()
top_percentage = category_percentage.max()

if top_percentage >= 40:
    archetype = "Shopping-Heavy Spender"
elif top_percentage >= 30:
    archetype = "Category-Focused Spender"
elif category_percentage.get('Food & Dining', 0) >= 25:
    archetype = "Food-Focused Spender"
else:
    archetype = "Balanced Spender"

print("=" * 60)
print("             SPENDING ARCHETYPE")
print("=" * 60)

print(f"Top Category : {top_category}")
print(f"Category Share : {top_percentage:.2f}%")
print(f"Archetype : {archetype}")

print("=" * 60)

             SPENDING ARCHETYPE
Top Category : Shopping
Category Share : 43.38%
Archetype : Shopping-Heavy Spender


In [ ]:
# ============================================================
# FEATURE 8 - SPENDING ARCHETYPE DETECTION
# ============================================================

print("=" * 70)
print("              SPENDING ARCHETYPE DETECTION")
print("=" * 70)

debits = df[df['type_clean'] == 'debit'].copy()

total_spending = debits['amount'].sum()

# ------------------------------------------------------------
# HELPER FUNCTION
# ------------------------------------------------------------

def percentage_of_spending(amount):
    if total_spending == 0:
        return 0
    return (amount / total_spending) * 100


# ------------------------------------------------------------
# 1. THE FOODIE
# Food Delivery + Restaurants + Cafe > 25%
# ------------------------------------------------------------

food_delivery_vendors = [
    'Swiggy',
    'Zomato'
]

restaurant_vendors = [
    'Restaurant'
]

cafe_vendors = [
    'Starbucks',
    'Third Wave Coffee',
    'Cafe Coffee Day',
    'Truffles'
]

foodie_vendors = (
    food_delivery_vendors +
    restaurant_vendors +
    cafe_vendors
)

foodie_spending = debits[
    debits['vendor_clean'].isin(foodie_vendors)
]['amount'].sum()

foodie_percentage = percentage_of_spending(foodie_spending)


# ------------------------------------------------------------
# 2. THE QUICK COMMERCE JUNKIE
# Quick Commerce > 15%
# ------------------------------------------------------------

quick_commerce_vendors = [
    'Instamart',
    'Blinkit',
    'Zepto',
    'BigBasket',
    'Grofers'
]

quick_commerce_spending = debits[
    debits['vendor_clean'].isin(quick_commerce_vendors)
]['amount'].sum()

quick_commerce_percentage = percentage_of_spending(
    quick_commerce_spending
)


# ------------------------------------------------------------
# 3. THE SHOPAHOLIC
# E-commerce > 15%
# ------------------------------------------------------------

ecommerce_vendors = [
    'Amazon',
    'Flipkart',
    'Myntra',
    'Nykaa'
]

ecommerce_spending = debits[
    debits['vendor_clean'].isin(ecommerce_vendors)
]['amount'].sum()

ecommerce_percentage = percentage_of_spending(
    ecommerce_spending
)


# ------------------------------------------------------------
# 4. THE INVESTOR
# Investments > 15%
# ------------------------------------------------------------

investment_spending = debits[
    debits['category'] == 'Investment'
]['amount'].sum()

investment_percentage = percentage_of_spending(
    investment_spending
)


# ------------------------------------------------------------
# 5. THE LATE-NIGHT SNACKER
# >50% of Food Delivery transactions between 21:00-02:00
# ------------------------------------------------------------

food_delivery = debits[
    debits['vendor_clean'].isin(food_delivery_vendors)
].copy()

late_night_food = food_delivery[
    (food_delivery['hour'] >= 21) |
    (food_delivery['hour'] < 2)
]

if len(food_delivery) > 0:
    late_night_percentage = (
        len(late_night_food) /
        len(food_delivery)
    ) * 100
else:
    late_night_percentage = 0


# ------------------------------------------------------------
# 6. THE CAB COMMUTER
# Transport > 10%
# ------------------------------------------------------------

transport_spending = debits[
    debits['category'] == 'Transport'
]['amount'].sum()

transport_percentage = percentage_of_spending(
    transport_spending
)


# ------------------------------------------------------------
# 7. THE SUBSCRIPTION LOVER
# 5 or more distinct subscription vendors
# ------------------------------------------------------------

subscription_vendors = [
    'Netflix',
    'Spotify',
    'Disney Hotstar',
    'Jio',
    'Airtel',
    'Vi'
]

active_subscription_vendors = debits[
    debits['vendor_clean'].isin(subscription_vendors)
]['vendor_clean'].nunique()

# ------------------------------------------------------------
# 8. SAVINGS ARCHETYPES
# ------------------------------------------------------------

total_income = df[
    df['type_clean'] == 'credit'
]['amount'].sum()

net_savings = total_income - total_spending

if total_income > 0:
    savings_rate = (
        net_savings / total_income
    ) * 100
else:
    savings_rate = 0


# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("\nARCHETYPE ANALYSIS")
print("-" * 70)

print(
    f"Food & Dining combination : "
    f"{foodie_percentage:.2f}%"
)

print(
    f"Quick Commerce spending   : "
    f"{quick_commerce_percentage:.2f}%"
)

print(
    f"E-commerce spending       : "
    f"{ecommerce_percentage:.2f}%"
)

print(
    f"Investment spending       : "
    f"{investment_percentage:.2f}%"
)

print(
    f"Late-night Food Delivery  : "
    f"{late_night_percentage:.2f}%"
)

print(
    f"Transport spending        : "
    f"{transport_percentage:.2f}%"
)

print(
    f"Subscription vendors      : "
    f"{active_subscription_vendors}"
)

print(
    f"Savings rate              : "
    f"{savings_rate:.2f}%"
)


# ------------------------------------------------------------
# DETECT ARCHETYPES
# ------------------------------------------------------------

print("\nDETECTED ARCHETYPES")
print("-" * 70)

archetype_found = False


# 1 Foodie
if foodie_percentage > 25:
    print(
        f"-> THE FOODIE "
        f"({foodie_percentage:.2f}% on Food Delivery + Restaurants + Cafe)"
    )
    archetype_found = True


# 2 Quick Commerce Junkie
if quick_commerce_percentage > 15:
    print(
        f"-> THE QUICK COMMERCE JUNKIE "
        f"({quick_commerce_percentage:.2f}% on Quick Commerce)"
    )
    archetype_found = True


# 3 Shopaholic
if ecommerce_percentage > 15:
    print(
        f"-> THE SHOPAHOLIC "
        f"({ecommerce_percentage:.2f}% on E-commerce)"
    )
    archetype_found = True


# 4 Investor
if investment_percentage > 15:
    print(
        f"-> THE INVESTOR "
        f"({investment_percentage:.2f}% on Investments)"
    )
    archetype_found = True


# 5 Late-Night Snacker
if late_night_percentage > 50:
    print(
        f"-> THE LATE-NIGHT SNACKER "
        f"({late_night_percentage:.2f}% of Food Delivery transactions at night)"
    )
    archetype_found = True


# 6 Cab Commuter
if transport_percentage > 10:
    print(
        f"-> THE CAB COMMUTER "
        f"({transport_percentage:.2f}% on Transport)"
    )
    archetype_found = True


# 7 Subscription Lover
if active_subscription_vendors >= 5:
    print(
        f"-> THE SUBSCRIPTION LOVER "
        f"({active_subscription_vendors} active subscription vendors)"
    )
    archetype_found = True


# 8 YOLO Spender
if savings_rate < 10:
    print(
        f"-> THE YOLO SPENDER "
        f"(Savings rate: {savings_rate:.2f}%)"
    )
    archetype_found = True


# 9 Disciplined Saver
if savings_rate > 40:
    print(
        f"-> THE DISCIPLINED SAVER "
        f"(Savings rate: {savings_rate:.2f}%)"
    )
    archetype_found = True


if not archetype_found:
    print("No defined archetype detected.")


print("=" * 70)

              SPENDING ARCHETYPE DETECTION

ARCHETYPE ANALYSIS
----------------------------------------------------------------------
Food & Dining combination : 16.57%
Quick Commerce spending   : 6.46%
E-commerce spending       : 35.97%
Investment spending       : 14.78%
Late-night Food Delivery  : 20.54%
Transport spending        : 3.42%
Subscription vendors      : 6
Savings rate              : -229.34%

DETECTED ARCHETYPES
----------------------------------------------------------------------
-> THE SHOPAHOLIC (35.97% on E-commerce)
-> THE SUBSCRIPTION LOVER (6 active subscription vendors)
-> THE YOLO SPENDER (Savings rate: -229.34%)


In [ ]:
print(df.columns.tolist())

['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode', 'Ref', 'date', 'amount', 'type_clean', 'vendor_clean', 'category', 'month', 'time_period']


In [ ]:
print(df['vendor_clean'].value_counts().head(15))
print("\nUnique vendors:", df['vendor_clean'].nunique())

Series([], Name: count, dtype: int64)

Unique vendors: 0


In [ ]:
# FEATURE 2 - VENDOR EXTRACTION

df['vendor_clean'] = df['Description'].apply(extract_vendor)

print("Unique Vendors:", df['vendor_clean'].nunique())
print("\nTop Vendors:")
print(df['vendor_clean'].value_counts().head(15))

Unique Vendors: 0

Top Vendors:
Series([], Name: count, dtype: int64)


In [ ]:
print(extract_vendor(df['Description'].iloc[0]))
print(df['Description'].iloc[0])

None
AMAZON SELLER SVCS


In [ ]:
# FEATURE 2 - VENDOR EXTRACTOR

vendor_keywords = {
    'Amazon': ['AMAZON', 'AMZN'],
    'Swiggy': ['SWIGGY', 'BUNDL TECH'],
    'Zomato': ['ZOMATO'],
    'Blinkit': ['BLINKIT'],
    'Zepto': ['ZEPTO', 'KIRANAKART'],
    'BigBasket': ['BIGBASKET'],
    'Grofers': ['GROFERS'],
    'Flipkart': ['FLIPKART', 'FKART'],
    'Myntra': ['MYNTRA'],
    'Nykaa': ['NYKAA', 'FSN E-COMMERCE'],
    'Uber': ['UBER', 'ANI TECHNOLOGIES'],
    'Ola': ['OLA CABS', 'OLA-PRIME', 'UPI-OLACABS'],
    'Rapido': ['RAPIDO'],
    'BMTC': ['BMTC', 'TUMMOC'],
    'Starbucks': ['STARBUCKS', 'TATA STARBUCKS'],
    'Third Wave Coffee': ['THIRDWAVE', 'THIRD WAVE', 'TWC INDIA'],
    'Cafe Coffee Day': ['COFFEE DAY', 'CCD'],
    'Truffles': ['TRUFFLES'],
    'DMart': ['DMART', 'AVENUE SUPERMARTS'],
    'HP Petrol': ['HP PETROL'],
    'BPCL': ['BPCL'],
    'Indian Oil': ['INDIAN OIL', 'IOC'],
    'Jio': ['RELIANCE JIO', 'JIORECHARGE', 'JIOFIBER'],
    'Airtel': ['AIRTEL', 'BHARTI AIRTEL'],
    'Vi': ['VODAFONE IDEA', 'VI POSTPAID', 'UPI-VI-RECHARGE'],
    'Netflix': ['NETFLIX'],
    'Spotify': ['SPOTIFY'],
    'Disney Hotstar': ['HOTSTAR', 'DISNEY HOTSTAR'],
    'BookMyShow': ['BOOKMYSHOW', 'BMS MOVIE TICKETS'],
    'Groww': ['GROWW'],
    'Zerodha': ['ZERODHA'],
    'BESCOM': ['BESCOM', 'BANGALORE ELEC SUPPLY'],
    'BWSSB': ['BWSSB', 'BWSSW'],
}

def extract_vendor(description):

    text = str(description).upper().strip()

    if 'ATM-WDL' in text:
        return 'Cash Withdrawal'

    if 'TECHCRUSH LABS-SALARY' in text:
        return 'Salary'

    if 'RENT-LANDLORD' in text:
        return 'Rent'

    if 'ROPPEN TRANSPORTATION' in text:
        return 'Transportation'

    if 'INNOVATIVE RETAIL' in text:
        return 'Innovative Retail'

    if 'STAR INDIA PVT LTD' in text:
        return 'Star India'

    if 'BIGTREE ENTERTAINMENT' in text:
        return 'Bigtree Entertainment'

    for person in [
        'AMAN', 'ANKIT', 'PRIYA', 'NEHA',
        'VIKAS', 'SNEHA', 'KARAN'
    ]:
        if person in text:
            return 'P2P Transfer'

    for vendor, keywords in vendor_keywords.items():
        for keyword in keywords:
            if keyword in text:
                return vendor

    if 'RESTAURANT' in text:
        return 'Restaurant'

    return 'Uncategorised'


# Apply vendor extraction
df['vendor_clean'] = df['Description'].apply(extract_vendor)

print("Unique Vendors:", df['vendor_clean'].nunique())

print("\nTop Vendors:")
print(df['vendor_clean'].value_counts().head(15))

Unique Vendors: 43

Top Vendors:
vendor_clean
Swiggy               223
Zomato               121
Uber                  89
Amazon                86
Zepto                 71
Uncategorised         62
Ola                   49
Flipkart              47
Starbucks             42
Restaurant            42
Rapido                41
Blinkit               40
BMTC                  37
Third Wave Coffee     31
Cafe Coffee Day       26
Name: count, dtype: int64


In [ ]:
print(df[df['vendor_clean'] == 'Uncategorised']['Description'].value_counts())

Description
OLA ELECTRIC           20
INSTAMART BANGALORE    16
POS MEGHANA FOODS      12
POS DINEOUT            10
POS INSTAMART           4
Name: count, dtype: int64


In [ ]:
def extract_vendor(description):

    text = str(description).upper().strip()

    if 'ATM-WDL' in text:
        return 'Cash Withdrawal'

    if 'TECHCRUSH LABS-SALARY' in text:
        return 'Salary'

    if 'RENT-LANDLORD' in text:
        return 'Rent'

    if 'OLA ELECTRIC' in text:
        return 'Ola Electric'

    if 'INSTAMART' in text:
        return 'Instamart'

    if 'MEGHANA FOODS' in text:
        return 'Restaurant'

    if 'DINEOUT' in text:
        return 'Restaurant'

    if 'ROPPEN TRANSPORTATION' in text:
        return 'Transportation'

    if 'INNOVATIVE RETAIL' in text:
        return 'Innovative Retail'

    if 'STAR INDIA PVT LTD' in text:
        return 'Star India'

    if 'BIGTREE ENTERTAINMENT' in text:
        return 'Bigtree Entertainment'

    for person in [
        'AMAN', 'ANKIT', 'PRIYA', 'NEHA',
        'VIKAS', 'SNEHA', 'KARAN'
    ]:
        if person in text:
            return 'P2P Transfer'

    for vendor, keywords in vendor_keywords.items():
        for keyword in keywords:
            if keyword in text:
                return vendor

    if 'RESTAURANT' in text:
        return 'Restaurant'

    return 'Uncategorised'


# Apply updated vendor extractor
df['vendor_clean'] = df['Description'].apply(extract_vendor)

print("Unique Vendors:", df['vendor_clean'].nunique())

print("\nUncategorised:")
print(df[df['vendor_clean'] == 'Uncategorised']['Description'].value_counts())

Unique Vendors: 44

Uncategorised:
Series([], Name: count, dtype: int64)


In [ ]:
# ============================================================
#                    SPENDDNA FINAL REPORT
# ============================================================

print("=" * 70)
print("                         SpendDNA")
print("              Personal Finance Analytics Report")
print("=" * 70)


# ============================================================
# [1] SPENDING OVERVIEW
# ============================================================

print("\n[1] SPENDING OVERVIEW")
print("-" * 70)

debit_df = df[df['type_clean'] == 'debit']
credit_df = df[df['type_clean'] == 'credit']

total_spending = debit_df['amount'].sum()
total_income = credit_df['amount'].sum()

transaction_count = len(df)
unique_vendors = df['vendor_clean'].nunique()

average_transaction = df['amount'].mean()
highest_transaction = df['amount'].max()

net_savings = total_income - total_spending

if total_income > 0:
    savings_rate = (net_savings / total_income) * 100
else:
    savings_rate = 0

print(f"Total Spending       : ₹{total_spending:,.2f}")
print(f"Total Income         : ₹{total_income:,.2f}")
print(f"Net Savings          : ₹{net_savings:,.2f}")
print(f"Savings Rate         : {savings_rate:.2f}%")
print(f"Transaction Count    : {transaction_count}")
print(f"Unique Vendors       : {unique_vendors}")
print(f"Average Transaction  : ₹{average_transaction:,.2f}")
print(f"Highest Transaction  : ₹{highest_transaction:,.2f}")


# ============================================================
# [2] CATEGORY-WISE SPENDING
# ============================================================

print("\n[2] CATEGORY-WISE SPENDING")
print("-" * 70)

category_spending = (
    debit_df.groupby('category')['amount']
    .sum()
    .sort_values(ascending=False)
)

for category, amount in category_spending.items():

    percentage = (
        amount / total_spending
    ) * 100

    print(
        f"{category:<20} : "
        f"₹{amount:,.2f} ({percentage:.2f}%)"
    )


# ============================================================
# [3] MONTHLY TREND ANALYSIS
# ============================================================

print("\n[3] MONTHLY TREND ANALYSIS")
print("-" * 70)

monthly_category = (
    debit_df
    .pivot_table(
        index='category',
        columns='month',
        values='amount',
        aggfunc='sum',
        fill_value=0
    )
    .sort_index(axis=1)
)

print("\nCategory × Month Spending:")
print(monthly_category.round(2).to_string())

monthly_total = (
    debit_df
    .groupby('month')['amount']
    .sum()
    .sort_index()
)

monthly_change = monthly_total.diff()

valid_change = monthly_change.dropna()

if len(valid_change) > 0:

    biggest_increase_month = valid_change.idxmax()
    biggest_increase = valid_change.max()

    biggest_decrease_month = valid_change.idxmin()
    biggest_decrease = valid_change.min()

    print(
        f"\nBiggest Increase : "
        f"{biggest_increase_month} "
        f"(₹{biggest_increase:,.2f})"
    )

    print(
        f"Biggest Decrease : "
        f"{biggest_decrease_month} "
        f"(₹{abs(biggest_decrease):,.2f})"
    )


# ============================================================
# [4] TIME-OF-DAY ANALYSIS
# ============================================================

print("\n[4] TIME-OF-DAY ANALYSIS")
print("-" * 70)

hourly_spending = (
    debit_df
    .groupby('hour')['amount']
    .sum()
    .sort_index()
)

print("\nHourly Spending:")

for hour, amount in hourly_spending.items():

    print(
        f"{hour:02d}:00 : "
        f"₹{amount:,.2f}"
    )

category_hour = (
    debit_df
    .pivot_table(
        index='category',
        columns='hour',
        values='amount',
        aggfunc='sum',
        fill_value=0
    )
    .sort_index(axis=1)
)

print("\nCategory × Hour Spending:")
print(category_hour.round(2).to_string())

highest_hour = hourly_spending.idxmax()
highest_hour_amount = hourly_spending.max()

print(
    f"\nPeak Spending Hour : "
    f"{highest_hour:02d}:00 "
    f"(₹{highest_hour_amount:,.2f})"
)

food_delivery_vendors = [
    'Swiggy',
    'Zomato'
]

food_delivery = debit_df[
    debit_df['vendor_clean'].isin(
        food_delivery_vendors
    )
]

late_night_food = food_delivery[
    (food_delivery['hour'] >= 21) |
    (food_delivery['hour'] < 2)
]

if len(food_delivery) > 0:

    late_night_percentage = (
        len(late_night_food) /
        len(food_delivery)
    ) * 100

else:

    late_night_percentage = 0

print(
    f"Late-Night Food Delivery : "
    f"{len(late_night_food)}/{len(food_delivery)} "
    f"({late_night_percentage:.2f}%)"
)


# ============================================================
# [5] ANOMALY DETECTION
# ============================================================

print("\n[5] ANOMALY DETECTION")
print("-" * 70)

print(f"Total Anomalies : {len(anomalies)}")
print("Method          : Within-category Z-score > 2")

if len(anomalies) > 0:

    print("\nTop 5 Anomalies:")

    top_anomalies = (
        anomalies
        .sort_values(
            'z_score',
            ascending=False
        )
        .head(5)
    )

    for _, row in top_anomalies.iterrows():

        print(
            f"{row['date'].date()} | "
            f"{row['Description'][:30]:<30} | "
            f"₹{row['amount']:,.2f} | "
            f"Z-score: {row['z_score']:.2f}"
        )


# ============================================================
# [6] SPENDING ARCHETYPES
# ============================================================

print("\n[6] SPENDING ARCHETYPES")
print("-" * 70)

print("\nArchetype Analysis:")

print(
    f"Food & Dining combination : "
    f"{foodie_percentage:.2f}%"
)

print(
    f"Quick Commerce spending   : "
    f"{quick_commerce_percentage:.2f}%"
)

print(
    f"E-commerce spending       : "
    f"{ecommerce_percentage:.2f}%"
)

print(
    f"Investment spending       : "
    f"{investment_percentage:.2f}%"
)

print(
    f"Late-night Food Delivery  : "
    f"{late_night_percentage:.2f}%"
)

print(
    f"Transport spending        : "
    f"{transport_percentage:.2f}%"
)

print(
    f"Subscription vendors      : "
    f"{active_subscription_vendors}"
)

print(
    f"Savings rate              : "
    f"{savings_rate:.2f}%"
)

print("\nDetected Archetypes:")

if foodie_percentage > 25:
    print(
        f"-> THE FOODIE "
        f"({foodie_percentage:.2f}%)"
    )

if quick_commerce_percentage > 15:
    print(
        f"-> THE QUICK COMMERCE JUNKIE "
        f"({quick_commerce_percentage:.2f}%)"
    )

if ecommerce_percentage > 15:
    print(
        f"-> THE SHOPAHOLIC "
        f"({ecommerce_percentage:.2f}% E-commerce)"
    )

if investment_percentage > 15:
    print(
        f"-> THE INVESTOR "
        f"({investment_percentage:.2f}%)"
    )

if late_night_percentage > 50:
    print(
        f"-> THE LATE-NIGHT SNACKER "
        f"({late_night_percentage:.2f}%)"
    )

if transport_percentage > 10:
    print(
        f"-> THE CAB COMMUTER "
        f"({transport_percentage:.2f}%)"
    )

if active_subscription_vendors >= 5:
    print(
        f"-> THE SUBSCRIPTION LOVER "
        f"({active_subscription_vendors} vendors)"
    )

if savings_rate < 10:
    print(
        f"-> THE YOLO SPENDER "
        f"(Savings rate: {savings_rate:.2f}%)"
    )

if savings_rate > 40:
    print(
        f"-> THE DISCIPLINED SAVER "
        f"(Savings rate: {savings_rate:.2f}%)"
    )


# ============================================================
# [7] KEY INSIGHTS
# ============================================================

print("\n[7] KEY INSIGHTS")
print("-" * 70)

top_category = category_spending.idxmax()
top_category_amount = category_spending.max()

top_category_percentage = (
    top_category_amount /
    total_spending
) * 100

print(
    f"1. {top_category} is the largest spending category "
    f"with {top_category_percentage:.2f}% of total spending."
)

print(
    f"2. {len(anomalies)} transactions were identified "
    f"as anomalies using a within-category Z-score > 2."
)

print(
    f"3. Peak spending hour is "
    f"{highest_hour:02d}:00 with "
    f"₹{highest_hour_amount:,.2f}."
)

print(
    f"4. The calculated savings rate is "
    f"{savings_rate:.2f}%."
)

print(
    f"5. The dataset contains "
    f"{unique_vendors} unique vendors across "
    f"{transaction_count} transactions."
)

print(
    f"6. Biggest monthly spending increase occurred in "
    f"{biggest_increase_month}."
)

print(
    f"7. Biggest monthly spending decrease occurred in "
    f"{biggest_decrease_month}."
)

print(
    f"8. Late-night Food Delivery transactions account for "
    f"{late_night_percentage:.2f}% of Food Delivery transactions."
)


# ============================================================
# [8] DATASET VALIDATION NOTE
# ============================================================

print("\n[8] DATASET VALIDATION")
print("-" * 70)

print(f"Final Rows              : {len(df)}")
print(f"Debit Transactions      : {len(debit_df)}")
print(f"Credit Transactions     : {len(credit_df)}")
print(f"Duplicate Rows Remaining: {df.duplicated().sum()}")
print(f"Invalid Transaction Types: {len(invalid_types)}")
print(f"Missing Amounts         : {df['amount'].isna().sum()}")

print("\nAmount Reconciliation:")

print(
    f"Debit + Credit : "
    f"₹{total_spending + total_income:,.2f}"
)

print(
    f"All Amounts    : "
    f"₹{df['amount'].sum():,.2f}"
)


# ============================================================
# END
# ============================================================

print("\n" + "=" * 70)
print("                  END OF SpendDNA REPORT")
print("=" * 70)

                         SpendDNA
              Personal Finance Analytics Report

[1] SPENDING OVERVIEW
----------------------------------------------------------------------
Total Spending       : ₹1,678,901.00
Total Income         : ₹509,774.00
Net Savings          : ₹-1,169,127.00
Savings Rate         : -229.34%
Transaction Count    : 1310
Unique Vendors       : 44
Average Transaction  : ₹1,670.74
Highest Transaction  : ₹85,492.00

[2] CATEGORY-WISE SPENDING
----------------------------------------------------------------------
Shopping             : ₹728,338.00 (43.38%)
Food & Dining        : ₹306,642.00 (18.26%)
Investment           : ₹248,160.00 (14.78%)
Rent                 : ₹108,000.00 (6.43%)
Fuel                 : ₹89,303.00 (5.32%)
Transport            : ₹57,474.00 (3.42%)
Cash Withdrawal      : ₹45,500.00 (2.71%)
Utilities            : ₹41,914.00 (2.50%)
Entertainment        : ₹26,764.00 (1.59%)
P2P Transfer         : ₹24,599.00 (1.47%)
Other                : ₹2,207.00 (0

In [ ]:
# ============================================================
# FEATURE 5 - MONTHLY TREND ANALYSIS
# ============================================================

print("=" * 70)
print("                 MONTHLY TREND ANALYSIS")
print("=" * 70)

# Debit transactions only
monthly_category = (
    df[df['type_clean'] == 'debit']
    .pivot_table(
        index='category',
        columns='month',
        values='amount',
        aggfunc='sum',
        fill_value=0
    )
)

# Sort months correctly
monthly_category = monthly_category.sort_index(axis=1)

print("\nCATEGORY-WISE MONTHLY SPENDING")
print("-" * 70)

print(monthly_category.round(2))


# ------------------------------------------------------------
# MONTH-TO-MONTH TOTAL SPENDING
# ------------------------------------------------------------

monthly_total = (
    df[df['type_clean'] == 'debit']
    .groupby('month')['amount']
    .sum()
    .sort_index()
)

print("\nMONTHLY TOTAL SPENDING")
print("-" * 70)

for month, amount in monthly_total.items():
    print(f"{month} : ₹{amount:,.2f}")


# ------------------------------------------------------------
# MONTH-ON-MONTH CHANGE
# ------------------------------------------------------------

print("\nMONTH-ON-MONTH CHANGE")
print("-" * 70)

monthly_change = monthly_total.diff()

for month, change in monthly_change.items():

    if pd.isna(change):
        print(f"{month} : N/A")
    elif change > 0:
        print(f"{month} : Increase of ₹{change:,.2f}")
    elif change < 0:
        print(f"{month} : Decrease of ₹{abs(change):,.2f}")
    else:
        print(f"{month} : No change")


# ------------------------------------------------------------
# BIGGEST INCREASE / DECREASE
# ------------------------------------------------------------

valid_change = monthly_change.dropna()

if len(valid_change) > 0:

    biggest_increase_month = valid_change.idxmax()
    biggest_increase = valid_change.max()

    biggest_decrease_month = valid_change.idxmin()
    biggest_decrease = valid_change.min()

    print("\nTREND SUMMARY")
    print("-" * 70)

    print(
        f"Biggest Increase : {biggest_increase_month} "
        f"(₹{biggest_increase:,.2f})"
    )

    print(
        f"Biggest Decrease : {biggest_decrease_month} "
        f"(₹{abs(biggest_decrease):,.2f})"
    )

print("=" * 70)

                 MONTHLY TREND ANALYSIS

CATEGORY-WISE MONTHLY SPENDING
----------------------------------------------------------------------
month             2024-01   2024-02   2024-03  2024-04   2024-05   2024-06
category                                                                  
Cash Withdrawal    2000.0    5000.0    8000.0   5500.0    8000.0   17000.0
Entertainment      4030.0    5102.0    5927.0   4392.0    1844.0    5469.0
Food & Dining     43344.0   52503.0   58835.0  43858.0   54351.0   53751.0
Fuel              30322.0    2079.0   26164.0  18718.0    9138.0    2882.0
Investment        38432.0   15000.0   68644.0  54126.0   48628.0   23330.0
Other               358.0       0.0     844.0    509.0     201.0     295.0
P2P Transfer       7852.0    4285.0    4275.0    663.0    3412.0    4112.0
Rent              18000.0   18000.0   18000.0  18000.0   18000.0   18000.0
Shopping         126267.0  115041.0  127415.0  90501.0  116251.0  152863.0
Transport         11005.0   1019

In [ ]:
# ============================================================
# FEATURE 6 - TIME-OF-DAY ANALYSIS
# ============================================================

print("=" * 70)
print("                 TIME-OF-DAY ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# CREATE HOUR COLUMN
# ------------------------------------------------------------

def extract_hour(time_value):
    try:
        return int(str(time_value).split(':')[0])
    except:
        return -1

df['hour'] = df['Time'].apply(extract_hour)


# ------------------------------------------------------------
# HOUR-WISE SPENDING
# ------------------------------------------------------------

hourly_spending = (
    df[df['type_clean'] == 'debit']
    .groupby('hour')['amount']
    .sum()
    .sort_index()
)

print("\nHOUR-WISE SPENDING")
print("-" * 70)

for hour, amount in hourly_spending.items():
    print(f"{hour:02d}:00 - ₹{amount:,.2f}")


# ------------------------------------------------------------
# CATEGORY × HOUR MATRIX
# ------------------------------------------------------------

category_hour = (
    df[df['type_clean'] == 'debit']
    .pivot_table(
        index='category',
        columns='hour',
        values='amount',
        aggfunc='sum',
        fill_value=0
    )
)

category_hour = category_hour.sort_index(axis=1)

print("\nCATEGORY × HOUR SPENDING")
print("-" * 70)

print(category_hour.round(2))


# ------------------------------------------------------------
# LATE-NIGHT FOOD DELIVERY
# 21:00 to 02:00
# ------------------------------------------------------------

food_delivery_vendors = [
    'Swiggy',
    'Zomato'
]

food_delivery = df[
    (df['type_clean'] == 'debit') &
    (df['vendor_clean'].isin(food_delivery_vendors))
].copy()

late_night_food = food_delivery[
    (food_delivery['hour'] >= 21) |
    (food_delivery['hour'] < 2)
]

if len(food_delivery) > 0:

    late_night_percentage = (
        len(late_night_food) /
        len(food_delivery)
    ) * 100

else:

    late_night_percentage = 0


print("\nLATE-NIGHT FOOD DELIVERY")
print("-" * 70)

print(
    f"Total Food Delivery Transactions : "
    f"{len(food_delivery)}"
)

print(
    f"Late-Night Transactions (21:00-02:00) : "
    f"{len(late_night_food)}"
)

print(
    f"Late-Night Percentage : "
    f"{late_night_percentage:.2f}%"
)


# ------------------------------------------------------------
# HIGHEST SPENDING HOUR
# ------------------------------------------------------------

if len(hourly_spending) > 0:

    highest_hour = hourly_spending.idxmax()
    highest_hour_amount = hourly_spending.max()

    print("\nPEAK SPENDING HOUR")
    print("-" * 70)

    print(
        f"{highest_hour:02d}:00 "
        f"with ₹{highest_hour_amount:,.2f}"
    )

print("=" * 70)

                 TIME-OF-DAY ANALYSIS

HOUR-WISE SPENDING
----------------------------------------------------------------------
00:00 - ₹37,969.00
01:00 - ₹38,882.00
02:00 - ₹20,283.00
03:00 - ₹34,868.00
04:00 - ₹43,901.00
05:00 - ₹33,671.00
06:00 - ₹55,506.00
07:00 - ₹19,938.00
08:00 - ₹41,399.00
09:00 - ₹86,701.00
10:00 - ₹140,526.00
11:00 - ₹123,389.00
12:00 - ₹77,096.00
13:00 - ₹113,819.00
14:00 - ₹121,010.00
15:00 - ₹39,647.00
16:00 - ₹100,597.00
17:00 - ₹56,604.00
18:00 - ₹101,343.00
19:00 - ₹67,848.00
20:00 - ₹138,089.00
21:00 - ₹81,545.00
22:00 - ₹55,867.00
23:00 - ₹48,403.00

CATEGORY × HOUR SPENDING
----------------------------------------------------------------------
hour                  0        1        2        3        4        5   \
category                                                                
Cash Withdrawal      0.0      0.0      0.0      0.0      0.0      0.0   
Entertainment      562.0   1658.0   1295.0   4029.0   4685.0    745.0   
Food & Dining     3

In [ ]:
# ============================================================
# AUDIT - CHECK DEBIT / CREDIT / AMOUNT PARSING
# ============================================================

print("=" * 70)
print("              TRANSACTION PARSING AUDIT")
print("=" * 70)

# 1. Type values
print("\n[1] TRANSACTION TYPE CHECK")
print("-" * 70)
print(df['type_clean'].value_counts())

# 2. Amount summary
print("\n[2] AMOUNT SUMMARY")
print("-" * 70)
print(df['amount'].describe())

# 3. Debit total
debit_total = df[
    df['type_clean'] == 'debit'
]['amount'].sum()

# 4. Credit total
credit_total = df[
    df['type_clean'] == 'credit'
]['amount'].sum()

# 5. Count
debit_count = (
    df['type_clean'] == 'debit'
).sum()

credit_count = (
    df['type_clean'] == 'credit'
).sum()

print(f"Debit Transactions  : {debit_count}")
print(f"Credit Transactions : {credit_count}")

print(f"\nTotal Debit  : ₹{debit_total:,.2f}")
print(f"Total Credit : ₹{credit_total:,.2f}")

# 6. Invalid types
print("\n[3] INVALID TYPE CHECK")
print("-" * 70)

valid_types = ['debit', 'credit']

invalid_types = df[
    ~df['type_clean'].isin(valid_types)
]

print("Invalid Type Rows :", len(invalid_types))

if len(invalid_types) > 0:
    print(invalid_types[['Date', 'Description', 'Type', 'Amount']])

# 7. Missing / invalid amounts
print("\n[4] AMOUNT VALIDATION")
print("-" * 70)

print("Missing Amounts :", df['amount'].isna().sum())
print("Zero Amounts    :", (df['amount'] == 0).sum())
print("Negative Amounts:", (df['amount'] < 0).sum())

# 8. Very large transactions
print("\n[5] TOP 10 LARGEST TRANSACTIONS")
print("-" * 70)

top_transactions = df.sort_values(
    'amount',
    ascending=False
).head(10)

print(
    top_transactions[
        ['date', 'Description', 'Type', 'type_clean', 'amount']
    ].to_string(index=False)
)

# 9. Debit + Credit reconciliation
print("\n[6] RECONCILIATION")
print("-" * 70)

print(f"Debit + Credit = ₹{debit_total + credit_total:,.2f}")
print(f"All Amounts    = ₹{df['amount'].sum():,.2f}")

if round(debit_total + credit_total, 2) == round(df['amount'].sum(), 2):
    print("Status: PASS")
else:
    print("Status: CHECK REQUIRED")

print("=" * 70)

              TRANSACTION PARSING AUDIT

[1] TRANSACTION TYPE CHECK
----------------------------------------------------------------------
type_clean
debit     1304
credit       6
Name: count, dtype: int64

[2] AMOUNT SUMMARY
----------------------------------------------------------------------
count     1310.000000
mean      1670.744275
std       6306.346647
min         10.000000
25%        295.250000
50%        477.000000
75%       1009.750000
max      85492.000000
Name: amount, dtype: float64
Debit Transactions  : 1304
Credit Transactions : 6

Total Debit  : ₹1,678,901.00
Total Credit : ₹509,774.00

[3] INVALID TYPE CHECK
----------------------------------------------------------------------
Invalid Type Rows : 0

[4] AMOUNT VALIDATION
----------------------------------------------------------------------
Missing Amounts : 0
Zero Amounts    : 0
Negative Amounts: 0

[5] TOP 10 LARGEST TRANSACTIONS
----------------------------------------------------------------------
      date     

In [ ]:
# ============================================================
# FINAL DATASET VERIFICATION
# ============================================================

print("=" * 70)
print("             RAW DATASET VERIFICATION")
print("=" * 70)

print("\nRAW DATA SHAPE")
print(df.shape)

print("\nORIGINAL TYPE VALUES")
print(df['Type'].value_counts())

print("\nDEBIT DESCRIPTION SAMPLE")
print("-" * 70)

print(
    df[df['type_clean'] == 'debit']
    [['Date', 'Description', 'Type', 'Amount']]
    .head(20)
    .to_string(index=False)
)

print("\nCREDIT DESCRIPTION SAMPLE")
print("-" * 70)

print(
    df[df['type_clean'] == 'credit']
    [['Date', 'Description', 'Type', 'Amount']]
    .to_string(index=False)
)

print("=" * 70)

             RAW DATASET VERIFICATION

RAW DATA SHAPE
(1310, 16)

ORIGINAL TYPE VALUES
Type
DR       662
Debit    642
CR         6
Name: count, dtype: int64

DEBIT DESCRIPTION SAMPLE
----------------------------------------------------------------------
       Date            Description  Type   Amount
 2024-01-01     AMAZON SELLER SVCS Debit    ₹2462
  01-Jan-24              BHIM-BMTC    DR    50.00
 2024-01-01   UPI-AMAN-8934@OKAXIS Debit    ₹1828
01 Jan 2024           BHIM-BLINKIT Debit   270.00
 2024-01-01             BHIM ZEPTO    DR  Rs. 625
 2024-01-01 UPI-UBER-2426@HDFCBANK Debit  Rs. 148
  01-Jan-24           BHIM-BLINKIT Debit     ₹482
 2024-01-02   POS SWIGGY BANGALORE Debit  Rs. 537
   02/01/24  UPI-GROWWPAY@HDFCBANK    DR     3956
   02/01/24           OLA ELECTRIC Debit     ₹350
  02-Jan-24      BMS MOVIE TICKETS Debit     ₹311
   02/01/24          POS OLA-PRIME    DR   258.00
02 Jan 2024       SWIGGY-INSTAMART Debit     ₹571
  02-Jan-24     UPI-STARBUCKS@AXIS Debit   266